In [35]:
import pandas as pd  
import numpy as np  
import os  
import re  
import glob  
import io
import calendar
from datetime import datetime  
import msoffcrypto
import warnings
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [36]:
import calendar  
import glob  
import io  
import os  
import re

import msoffcrypto  
import pandas as pd  
from msoffcrypto.exceptions import FileFormatError, InvalidKeyError


def read_protected_excel(file_path, password, sheet_names, skiprows_list):  
    """Read multiple sheets from an Excel file that may or may not be password-protected."""  
    with open(file_path, 'rb') as f:  
        raw = io.BytesIO(f.read())

    workbook = raw  
    encrypted = False

    try:  
        raw.seek(0)  
        office_file = msoffcrypto.OfficeFile(raw)  
        if office_file.is_encrypted():  
            encrypted = True  
            decrypted = io.BytesIO()  
            office_file.load_key(password=password)  
            office_file.decrypt(decrypted)  
            workbook = decrypted  
    except FileFormatError:  
        workbook = raw  
    except InvalidKeyError:  
        raise RuntimeError(f"Wrong password for {os.path.basename(file_path)}")

    print(f"    ({'encrypted' if encrypted else 'not encrypted'})")

    dfs = []  
    for sheet_name, skiprows in zip(sheet_names, skiprows_list):  
        workbook.seek(0)  
        df = pd.read_excel(workbook, sheet_name=sheet_name, skiprows=skiprows)  
        dfs.append(df)

    return dfs


def clean_columns(df):  
    df.columns = (  
        df.columns.astype(str)  
        .str.strip()  
        .str.replace(r'[^a-zA-Z0-9]', '_', regex=True)  
        .str.replace(r'_+', '_', regex=True)  
        .str.strip('_')  
        .str.lower()  
    )  
    return df


# ---------------- config ----------------  
base_paths = [  
    'L:/2023 Pilar Plant Files/2023 Manpower Detail/',  
    'L:/2024 Pilar Plant Files/2024 Manpower Detail/',  
    'L:/2025 Pilar Plant Files/2025 Manpower Detail/',  
    'L:/2026 Pilar Plant Files/2026 Manpower Detail/',  
]  
password = 'bamhorse2'  
sheet_names = ['YTD FTEs', 'YTD Dollars']  
skiprows_list = [7, 7]  
required_names = ['LENOX SAL']  
start_date = pd.to_datetime('2026-09-30')

# ---------------- collect files across ALL paths ----------------  
all_files = []  
for base_path in base_paths:  
    if not os.path.isdir(base_path):  
        print(f"WARNING: path not found, skipping -> {base_path}")  
        continue  
    found = glob.glob(os.path.join(base_path, '*.xlsm'))  
    print(f"{len(found):4d} .xlsm files in {base_path}")  
    all_files.extend(found)

# ---------------- filter ----------------  
month_end_files = []  
seen = set()

for f in all_files:  
    filename = os.path.basename(f)

    if filename.startswith('~$'):          # Excel lock/temp file  
        continue

    upper_name = filename.upper()

    matched_name = next(  
        (name for name in required_names if name.upper() in upper_name), None  
    )  
    if matched_name is None:  
        continue

    site_code = matched_name.split(' ')[0]

    date_match = re.search(r'(\d{4}-\d{2}-\d{2})', filename)  
    if not date_match:  
        print(f"  WARNING: Could not parse date from {filename}")  
        continue

    date_str = date_match.group(1)  
    file_date = pd.to_datetime(date_str, errors='coerce')  
    if pd.isna(file_date):  
        print(f"  WARNING: Bad date '{date_str}' in {filename}")  
        continue

    if file_date < start_date:  
        continue

    last_day = calendar.monthrange(file_date.year, file_date.month)[1]  
    if file_date.day != last_day:  
        continue

    key = (site_code, date_str)            # dedupe same site+month across folders  
    if key in seen:  
        print(f"  DUPLICATE skipped: {filename}")  
        continue  
    seen.add(key)

    month_end_files.append((f, date_str, site_code))

month_end_files.sort(key=lambda x: (x[1], x[2]))

print(f"\nMonth-end files found (on/after {start_date.date()}): {len(month_end_files)}")  
for f, d, name in month_end_files:  
    print(f"  {os.path.basename(f)} (date: {d} | site: {name})")

if not month_end_files:  
    raise SystemExit("No month-end files matched. Check base_paths / required_names / start_date.")

# ---------------- read ----------------  
files = [(f, 'Monthly', d, name) for f, d, name in month_end_files]

all_mtd_ftes = []  
all_mtd_doll = []  
failed = []

for file_path, pay_type, date, site in files:  
    print(f"\nReading {pay_type} | Site: {site} | Date: {date}")  
    try:  
        dfs = read_protected_excel(file_path, password, sheet_names, skiprows_list)  
        for df in dfs:  
            df['Pay_Type'] = pay_type  
            df['Report_Date'] = pd.to_datetime(date)  
            df['BU'] = site  
            df['Source_File'] = os.path.basename(file_path)

        all_mtd_ftes.append(clean_columns(dfs[0]))  
        all_mtd_doll.append(clean_columns(dfs[1]))

    except Exception as e:  
        print(f"  ERROR: {type(e).__name__}: {e}")  
        failed.append((os.path.basename(file_path), str(e)))

if not all_mtd_ftes:  
    raise SystemExit("Every file failed to read. See errors above.")

mtd_ftes_combined = pd.concat(all_mtd_ftes, ignore_index=True)  
mtd_doll_combined = pd.concat(all_mtd_doll, ignore_index=True)

# ---------------- summary ----------------  
print("\n--- Summary ---")  
print(f"Files attempted: {len(files)} | succeeded: {len(all_mtd_ftes)} | failed: {len(failed)}")  
if failed:  
    for name, err in failed:  
        print(f"  FAILED: {name} -> {err}")  
print(f"MTD FTEs:    {mtd_ftes_combined.shape}")  
print(f"MTD Dollars: {mtd_doll_combined.shape}")  
print(f"Sites found: {sorted(mtd_doll_combined['bu'].unique())}")  
print(f"Date range:  {mtd_doll_combined['report_date'].min().date()} "  
      f"to {mtd_doll_combined['report_date'].max().date()}")  

 178 .xlsm files in L:/2023 Pilar Plant Files/2023 Manpower Detail/
 152 .xlsm files in L:/2024 Pilar Plant Files/2024 Manpower Detail/
 147 .xlsm files in L:/2025 Pilar Plant Files/2025 Manpower Detail/
 140 .xlsm files in L:/2026 Pilar Plant Files/2026 Manpower Detail/

Month-end files found (on/after 2026-09-30): 1
  LENOX SALAERN LENOX_EXEC 2026-09-30.xlsm (date: 2026-09-30 | site: LENOX)

Reading Monthly | Site: LENOX | Date: 2026-09-30
    (encrypted)

--- Summary ---
Files attempted: 1 | succeeded: 1 | failed: 0
MTD FTEs:    (1349, 21)
MTD Dollars: (1349, 21)
Sites found: ['LENOX']
Date range:  2026-09-30 to 2026-09-30


In [37]:
# function to clean tabs
def clean_sheet(sheet):
    # remove subtotal rowa
    sheet_no_totals = sheet[~sheet['deptid'].str.contains('total', case=False, na=False)]

    # forward fill department ids and names
    sheet_no_totals['deptid'] = sheet_no_totals['deptid'].ffill()
    sheet_no_totals['department_description'] = sheet_no_totals['department_description'].ffill()

    # remove department ids that are blank
    sheet_no_nulls = sheet_no_totals[sheet_no_totals['deptid'] != '(blank)']
    # set type to integer
    sheet_no_nulls[['deptid', 'job_code']] = sheet_no_nulls[['deptid', 'job_code']].astype('Int64') 

    # fix duplicate column names  
    cols = pd.Series(sheet_no_nulls.columns)  
    for dup in cols[cols.duplicated()].unique():  
        count = 0  
        for i in range(len(cols)):  
            if cols[i] == dup:  
                count += 1  
                if count > 1:  
                    cols[i] = f"{dup}_{count}"  
    sheet_no_nulls.columns = cols

    sheet_filtered = sheet_no_nulls[['report_date', 'bu', 'deptid', 'department_description', 'job_code', 'job_code_description',  
                                     'overtime', 'total', 'overtime_2', 'total_2']]
    
    sheet_calced = sheet_filtered.copy()
    
    is_flex = sheet_calced['job_code_description'].str.contains('flex', case=False, na=False)  
    not_flex = ~is_flex

    for suffix, tot, ot in [('a', 'total', 'overtime'), ('b', 'total_2', 'overtime_2')]:  
        sheet_calced[f'reg_{suffix}']   = (sheet_calced[tot] - sheet_calced[ot]).where(not_flex).fillna(0)
        sheet_calced[f'ot_{suffix}']    = sheet_calced[ot].where(not_flex).fillna(0)
        sheet_calced[f'flex_{suffix}']  = sheet_calced[tot].where(is_flex).fillna(0)
        sheet_calced[f'total_{suffix}'] = sheet_calced[tot].fillna(0)

    for metric in ['reg', 'ot', 'flex', 'total']:  
        sheet_calced[f'{metric}_v'] = sheet_calced[f'{metric}_b'] - sheet_calced[f'{metric}_a']  

    sheet_final = sheet_calced.drop(columns=['overtime', 'total', 'overtime_2', 'total_2'])

    return sheet_final

ftes_all = clean_sheet(mtd_ftes_combined)
dollars_all = clean_sheet(mtd_doll_combined)

In [38]:
vp_xwalk = pd.read_excel(  
    'L:/2026 Pilar Plant Files/Crosswalks/Manhattan Region Crosswalk.xlsx',  
    sheet_name='vp_dept'  
)

def spec_vp(sheet, vp=None):  
    sheet_vps = pd.merge(vp_xwalk, sheet, how='right', left_on='Dept', right_on='deptid').drop(columns=['Dept']).rename(columns={'VP': 'vp'})

    if vp is None:  
        return sheet_vps

    if vp == 'Browne, Kevin':  
        kevin_gv_depts = [74002506, 74002520, 74002561]  
        sheet_spec = sheet_vps[(sheet_vps['vp'] == vp) | (sheet_vps['deptid'].isin(kevin_gv_depts))]  
        sheet_kevin = sheet_spec[sheet_spec['deptid'] != 15600090]  
        return sheet_kevin
    elif vp == 'Bedi, Anupama':
        anu_depts = [15612095, 15624050, 15625045, 15625045, 15628030, 15632045, 15633025, 15635545, 15635550, 15638045, 15638045, 15652045, 15652065]
        sheet_spec = sheet_vps[(sheet_vps['deptid'].isin(anu_depts))]  
        return sheet_spec
    else:  
        sheet_spec = sheet_vps[sheet_vps['vp'] == vp]  
        return sheet_spec  

ftes = spec_vp(ftes_all, 'Bedi, Anupama')
dollars = spec_vp(dollars_all, 'Bedi, Anupama')

file_name = 'acp_fte_dollars_summary_2026'

In [39]:

with pd.ExcelWriter(f'{file_name}.xlsx', engine='openpyxl') as writer: 
    ftes.to_excel(writer, sheet_name='MTD FTEs', index=False)  
    dollars.to_excel(writer, sheet_name='MTD Dollars', index=False)

print(f"Written to {file_name}.xlsx")

Written to acp_fte_dollars_summary_2026.xlsx
